# DEXA Data Unified Format

## Purpose
This notebook standardizes DEXA scan data across all 5 batches by:
- Loading individual batch CSV files
- Standardizing timepoint naming conventions
- Integrating image metadata with measurement data
- Exporting unified dataset for analysis and visualization

## Input Files
- batch1_dexa_cleaned.csv through batch5_dexa_cleaned.csv
- Image files (.jpg and .bmp) from batch directories

## Output Files
- unified_dexa_all_batches_with_images.csv
- unified_dexa_all_batches_with_images.xlsx (multi-sheet workbook)

## Setup and Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import os

# Configure paths
batch1_path = Path("/Users/aviado/Documents/GDG WashU Medicine/Sample Data/DEXA Scans/Batch 1")
batch2_path = Path("/Users/aviado/Documents/GDG WashU Medicine/Sample Data/DEXA Scans/Batch 2")
batch3_path = Path("/Users/aviado/Documents/GDG WashU Medicine/Sample Data/DEXA Scans/Batch 3")
batch4_path = Path("/Users/aviado/Documents/GDG WashU Medicine/Sample Data/DEXA Scans/Batch 4")
batch5_path = Path("/Users/aviado/Documents/GDG WashU Medicine/Sample Data/DEXA Scans/Batch 5")
output_dir = Path("cleaned_output")
output_dir.mkdir(exist_ok=True)

## Data Loading

Load all cleaned batch CSV files and remove duplicates from Batch 2.

In [ ]:
def load_all_batch_data(output_dir):
    """Load all cleaned batch CSV files and remove duplicates"""
    batch_data = {}
    
    for i in range(1, 6):
        batch_file = output_dir / f"batch{i}_dexa_cleaned.csv"
        
        if batch_file.exists():
            df = pd.read_csv(batch_file)
            
            # Remove duplicates for all batches
            original_count = len(df)
            df = df.drop_duplicates()
            deduplicated_count = len(df)
            
            if original_count != deduplicated_count:
                print(f"Batch {i}: Removed {original_count - deduplicated_count} duplicates")
            
            batch_data[f'batch{i}'] = df
    
    return batch_data

batch_data = load_all_batch_data(output_dir)

print("\nBatch summary:")
for batch_name, df in batch_data.items():
    print(f"{batch_name}: {len(df)} records")

total_records = sum(len(df) for df in batch_data.values())
print(f"\nTotal records: {total_records}")

## Timepoint Standardization

Standardize timepoint naming across batches:
- Batch 1: Week_0 → Baseline
- Batches 2-5: Pre_Scan → Baseline

In [ ]:
def standardize_timepoints(df):
    """Standardize timepoint names across all batches"""
    df_standardized = df.copy()
    
    timepoint_mapping = {
        'Week_0': 'Baseline',
        'Pre_Scan': 'Baseline',
        'Week_1': 'Week_1',
        'Week_2': 'Week_2',
        'Week_3': 'Week_3',
        'Post_Scan': 'Post_Scan',
        'Root': 'Unknown'
    }
    
    df_standardized['timepoint_standardized'] = df_standardized['timepoint'].map(timepoint_mapping)
    df_standardized['timepoint_original'] = df_standardized['timepoint']
    
    # Handle any unmapped timepoints
    unmapped = df_standardized['timepoint_standardized'].isnull()
    if unmapped.any():
        df_standardized.loc[unmapped, 'timepoint_standardized'] = df_standardized.loc[unmapped, 'timepoint_original']
    
    return df_standardized

# Combine all batches
all_batches_df = pd.concat([df for df in batch_data.values()], ignore_index=True)

# Apply standardization
all_batches_standardized = standardize_timepoints(all_batches_df)

print(f"Total records after combining: {len(all_batches_standardized)}")
print(f"\nTimepoint distribution by batch:")
print(all_batches_standardized.groupby(['batch', 'timepoint_standardized']).size().unstack(fill_value=0))

## Image Data Scanning

Scan all batch directories for DEXA scan images (.jpg and .bmp formats).

In [ ]:
def scan_dexa_images():
    """Scan for images across all DEXA batch directories"""
    all_images = []
    
    batch_configs = {
        'Batch_1': {
            'path': batch1_path,
            'timepoints': ['Week 0 DEXA', 'Week 1 DEXA', 'Week 2 DEXA', 'Week 3 DEXA (Named Week 4)', 'Post-Scan']
        },
        'Batch_2': {
            'path': batch2_path,
            'timepoints': ['Pre-Scan', 'WEEK1', 'Week2', 'Week 3', 'Post-Scan']
        },
        'Batch_3': {
            'path': batch3_path,
            'timepoints': ['Pre-Scan', '1 week post-treatment', '2 week post-treatment', '3 week post-treatment', 'Post-scan']
        },
        'Batch_4': {
            'path': batch4_path,
            'timepoints': ['Pre-Scan', '1 week post-treatment', '2 weeks post-treatment', '3 weeks post-treatment', 'Post-scan']
        },
        'Batch_5': {
            'path': batch5_path,
            'timepoints': ['Pre-Scan', '1 week post-treatment', '2 week post-treatment', '3 week post-treatment', 'Post-Scan']
        }
    }
    
    for batch_name, config in batch_configs.items():
        batch_path = config['path']
        
        for timepoint in config['timepoints']:
            timepoint_path = batch_path / timepoint
            if timepoint_path.exists():
                for gender in ['Male', 'Female']:
                    gender_path = timepoint_path / gender
                    if gender_path.exists():
                        for ext in ['*.jpg', '*.bmp']:
                            for img_file in gender_path.glob(ext):
                                all_images.append({
                                    'batch': batch_name,
                                    'timepoint_dir': timepoint,
                                    'gender': gender,
                                    'subject_id': img_file.stem.split()[0],
                                    'image_path': img_file,
                                    'filename': img_file.name
                                })
    
    return all_images

dexa_images = scan_dexa_images()
print(f"Total images found: {len(dexa_images)}")

## Image Metadata Extraction

Extract image dimensions and metadata for integration with DEXA measurements.

In [ ]:
def analyze_image_metadata(image_list):
    """Extract metadata from DEXA images"""
    image_data = []
    
    for img_info in image_list:
        try:
            with Image.open(img_info['image_path']) as img:
                image_data.append({
                    'batch': img_info['batch'],
                    'subject_id': img_info['subject_id'],
                    'timepoint_dir': img_info['timepoint_dir'],
                    'gender': img_info['gender'],
                    'width': img.width,
                    'height': img.height,
                    'aspect_ratio': img.width / img.height,
                    'file_size_mb': os.path.getsize(img_info['image_path']) / (1024*1024),
                    'image_path': str(img_info['image_path']),
                    'image_type': 'bmp' if img_info['filename'].endswith('.bmp') else 'jpg'
                })
        except Exception:
            continue
    
    return pd.DataFrame(image_data)

image_df = analyze_image_metadata(dexa_images)
print(f"Images processed: {len(image_df)}")
print(f"\nImage format distribution:")
print(image_df['image_type'].value_counts())

## Data Integration

Integrate image metadata with DEXA measurement data.

In [ ]:
def integrate_images_with_data(dexa_df, image_df):
    """Integrate image information with DEXA measurements"""
    
    # Map image timepoints to standardized names
    timepoint_mapping = {
        'Week 0 DEXA': 'Baseline',
        'Week 1 DEXA': 'Week_1',
        'Week 2 DEXA': 'Week_2',
        'Week 3 DEXA (Named Week 4)': 'Week_3',
        'Post-Scan': 'Post_Scan',
        'Pre-Scan': 'Baseline',
        'WEEK1': 'Week_1',
        'Week2': 'Week_2',
        'Week 3': 'Week_3',
        '1 week post-treatment': 'Week_1',
        '2 week post-treatment': 'Week_2',
        '2 weeks post-treatment': 'Week_2',
        '3 week post-treatment': 'Week_3',
        '3 weeks post-treatment': 'Week_3',
        'Post-scan': 'Post_Scan'
    }
    
    image_df['timepoint_standardized'] = image_df['timepoint_dir'].map(timepoint_mapping)
    
    # Aggregate images per DEXA record
    image_summary = image_df.groupby(['batch', 'subject_id', 'timepoint_standardized', 'gender']).agg({
        'image_path': lambda x: '; '.join([str(p) for p in x]),
        'image_type': lambda x: '; '.join(x),
        'aspect_ratio': 'mean',
        'width': 'mean',
        'height': 'mean',
        'file_size_mb': 'sum'
    }).reset_index()
    
    # Merge with DEXA data
    dexa_with_images = dexa_df.merge(
        image_summary,
        on=['batch', 'subject_id', 'timepoint_standardized', 'gender'],
        how='left'
    )
    
    # Fill missing values
    dexa_with_images['image_path'] = dexa_with_images['image_path'].fillna('No images found')
    dexa_with_images['image_type'] = dexa_with_images['image_type'].fillna('None')
    dexa_with_images['aspect_ratio'] = dexa_with_images['aspect_ratio'].fillna(0)
    
    return dexa_with_images

unified_data = integrate_images_with_data(all_batches_standardized, image_df)

print(f"Total records: {len(unified_data)}")
print(f"Records with images: {(unified_data['image_path'] != 'No images found').sum()}")

## Data Export

Export unified dataset to CSV and Excel formats.

In [ ]:
# Organize columns
column_order = [
    'batch', 'subject_id', 'timepoint_standardized', 'timepoint_original', 'gender', 'filename',
    'total_weight', 'soft_weight', 'lean_weight', 'fat_weight', 'fat_percent',
    'bmc', 'bmd', 'bone_area', 'sample_area',
    'image_path', 'image_type', 'aspect_ratio', 'width', 'height', 'file_size_mb'
]

available_cols = [col for col in column_order if col in unified_data.columns]
extra_cols = [col for col in unified_data.columns if col not in column_order]
unified_data_export = unified_data[available_cols + extra_cols]

# Export CSV
csv_path = output_dir / "unified_dexa_all_batches_with_images.csv"
unified_data_export.to_csv(csv_path, index=False)

# Export Excel with multiple sheets
excel_path = output_dir / "unified_dexa_all_batches_with_images.xlsx"

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Main data
    unified_data_export.to_excel(writer, sheet_name='Unified_DEXA_Data', index=False)
    
    # Summary by batch and timepoint
    summary = unified_data_export.groupby(['batch', 'timepoint_standardized']).agg({
        'subject_id': 'nunique',
        'total_weight': 'mean',
        'fat_percent': 'mean',
        'bmd': 'mean'
    }).round(3)
    summary.to_excel(writer, sheet_name='Summary')
    
    # Metadata
    metadata = pd.DataFrame([{
        'Total_Records': len(unified_data_export),
        'Unique_Subjects': unified_data_export['subject_id'].nunique(),
        'Batches': ', '.join(sorted(unified_data_export['batch'].unique())),
        'Timepoints': ', '.join(sorted(unified_data_export['timepoint_standardized'].unique())),
        'Records_With_Images': (unified_data_export['image_path'] != 'No images found').sum(),
        'Date_Created': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    }])
    metadata.to_excel(writer, sheet_name='Metadata', index=False)

print(f"\nExport complete:")
print(f"CSV: {csv_path}")
print(f"Excel: {excel_path}")
print(f"\nFinal record count: {len(unified_data_export)}")